# 🤖 Project Syn // Snap ML Random Forest Anomaly Detector
### *Pillar 2: Safety-Bounded Machine Learning on IBM LinuxONE s390x*
---
This notebook demonstrates **IBM Snap ML / Scikit-Learn Random Forest** classification trained on multi-variate telemetry streams (grid voltage, flood index, route congestion, and battery levels).
The model detects physical sensor spoofing, hospital ward depletion, and power failures in $< 0.8$ milliseconds without external cloud dependencies.


## 1. Load Data & Prepare Training Matrix


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import time
import pickle

np.random.seed(42)
n_samples = 15000

# Normal telemetry baseline
norm_voltage = np.random.normal(220.0, 4.0, int(n_samples * 0.85))
norm_flood = np.random.exponential(0.08, int(n_samples * 0.85))
norm_congestion = np.random.uniform(10.0, 55.0, int(n_samples * 0.85))
norm_labels = np.zeros(int(n_samples * 0.85))

# Anomaly telemetry (Monsoon inundation, brownouts, replay attacks)
anom_voltage = np.concatenate([np.random.uniform(140.0, 185.0, int(n_samples * 0.08)), np.random.uniform(255.0, 280.0, int(n_samples * 0.07))])
anom_flood = np.random.uniform(0.40, 1.20, int(n_samples * 0.15))
anom_congestion = np.random.uniform(70.0, 100.0, int(n_samples * 0.15))
anom_labels = np.ones(int(n_samples * 0.15))

X = np.column_stack([
    np.concatenate([norm_voltage, anom_voltage]),
    np.concatenate([norm_flood, anom_flood]),
    np.concatenate([norm_congestion, anom_congestion])
])
y = np.concatenate([norm_labels, anom_labels])

df = pd.DataFrame(X, columns=['grid_voltage', 'flood_index', 'route_congestion'])
df['is_anomaly'] = y
print(f"Generated {len(df):,} synthetic multivariate telemetry records.")
df.head()


## 2. Train Random Forest Classifier on s390x


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=50, max_depth=12, n_jobs=-1, random_state=42)

t0 = time.perf_counter()
rf.fit(X_train, y_train)
train_time_ms = (time.perf_counter() - t0) * 1000

print(f"Random Forest Model Training Complete in {train_time_ms:.2f} ms")


## 3. Real-Time Micro-Inference Latency & Model Performance


In [ ]:
# Benchmark single-packet inference latency
latencies = []
for i in range(1000):
    sample = X_test[i:i+1]
    t_start = time.perf_counter()
    _ = rf.predict(sample)
    latencies.append((time.perf_counter() - t_start) * 1000)

avg_inf_ms = np.mean(latencies)
p99_inf_ms = np.percentile(latencies, 99)

y_pred = rf.predict(X_test)
print(f"Average Inference Latency: {avg_inf_ms:.3f} ms (P99: {p99_inf_ms:.3f} ms)")
print(f"ROC AUC Score:              {roc_auc_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['NOMINAL', 'ANOMALY']))


## 4. Export Artifact for s390x Production Engine


In [ ]:
os.makedirs('artifacts', exist_ok=True)
with open('artifacts/snapml_hospital_anomaly.pkl', 'wb') as f:
    pickle.dump(rf, f)

print(f"Artifact successfully saved to: artifacts/snapml_hospital_anomaly.pkl")
print(f"Model file size: {os.path.getsize('artifacts/snapml_hospital_anomaly.pkl') / 1024:.1f} KB")
